# STATEMENT CREATION WORKFLOW

This notebook documents the full workflow used to construct the
statement-level dataset from raw news articles. The purpose of 
the workflow is to move from unstructured article text to a 
structured dataset of direct group statements, suitable for 
computational analysis of media voice, framing, and public 
interest advocacy.

## The workflow proceeds through five main stages:

## 1. Article Ingestion and Pre-processing
    Raw news articles are imported, cleaned, and standardised.
    This includes removing formatting artefacts, normalising
    dates, and retaining only relevant textual content for
    quote detection.

## 2. Quote Extraction
    Direct speech is identified using the ATAP quoation tool.

##   3. Speaker Identification and Coreference Resolution
    Extracted quotes are linked to speakers using named-entity
    recognition and rule-based coreference resolution. This step
    assigns each statement to a unique interest group where
    possible.

##   4. Statement Validation and Filtering
    Automated assignments are filtered and validated to remove
    misattributions, ambiguous speakers, and non-substantive
    quotations. The output of this step is a high-confidence
    statement dataset.

## 5. Enrichment with Surrounding Text
    Each quote is extracted together with its
    surrounding textual context to preserve interpretive meaning.

 The final output of this workflow is a statement-level dataset
 in which each row represents a single direct quotation
 attributed to a specific interest group. 



In [5]:
# ----------------------------------------------------------------
# STAGE 1: R–PYTHON INTEGRATION AND DATA LOADING (REPLICATION STEP)
# ----------------------------------------------------------------
#
# This stage initializes the hybrid R–Python workflow used in the
# statement creation pipeline. Interoperability between Python
# and R is established via the `rpy2` package, enabling direct
# execution of R scripts from within this notebook.
#
# INPUT:
# - An external R script located in the /proceeding directory
#   (e.g., `proceeding/load_and_preprocess_articles.R`)
# - Raw news article data accessed and processed by this script
#
# PROCESS:
# - The R script is sourced and executed via `rpy2`
# - Initial data loading and preprocessing are performed in R
# - Preprocessed article-level data objects are passed to the
#   Python environment for downstream analysis
#
# OUTPUT:
# - Preprocessed article-level dataset available as a Python
#   object for use in subsequent stages of the workflow
#
# This step ensures reproducibility by preserving the original
# R-based preprocessing logic while integrating it directly into
# the Python-based statement extraction pipeline.
# ----------------------------------------------------------------

In [6]:
# R packages are installed as part of the first-time setup.
# See the README for the full setup instructions, including the R install.packages() command.
# or if they are not in the R environment yet, uncomment and run the line below here:
# r('install.packages(c("readxl", "openxlsx", "tidyr", "dplyr", "stringdist", "lubridate", "readr"), repos="https://cloud.r-project.org")')

In [7]:
# Windows R initialisation — only runs on Windows; safely ignored on macOS/Linux.
# Locates R automatically via the Windows registry or common install paths,
# sets R_HOME, extends PATH, and registers the R DLL directories so that
# rpy2 can find and load R.  Run this cell BEFORE any rpy2 import.
import sys, os

if sys.platform == 'win32':
    import winreg, glob

    def _find_r_home_windows():
        # 1. Already set externally — trust it
        if os.environ.get('R_HOME') and os.path.isdir(os.environ['R_HOME']):
            return os.environ['R_HOME']
        # 2. Standard R installer writes the path to the registry
        for hive in (winreg.HKEY_LOCAL_MACHINE, winreg.HKEY_CURRENT_USER):
            for subkey in (r'SOFTWARE\R-core\R', r'SOFTWARE\R-core\R64'):
                try:
                    key = winreg.OpenKey(hive, subkey)
                    path, _ = winreg.QueryValueEx(key, 'InstallPath')
                    winreg.CloseKey(key)
                    if os.path.isdir(path):
                        return path
                except OSError:
                    pass
        # 3. Common install locations (newest version first)
        for pattern in [
            r'C:\Program Files\R\R-*',
            os.path.expandvars(r'%LOCALAPPDATA%\Programs\R\R-*'),
        ]:
            matches = sorted(glob.glob(pattern), reverse=True)
            if matches:
                return matches[0]
        return None

    r_home = _find_r_home_windows()

    if r_home is None:
        raise EnvironmentError(
            'Could not locate R on this machine.\n'
            'Set R_HOME before launching Jupyter, e.g. in PowerShell:\n'
            '  $env:R_HOME = "C:\\Users\\<you>\\AppData\\Local\\Programs\\R\\R-4.x.x"\n'
            'Then restart the kernel and rerun this cell.'
        )

    r_bin   = os.path.join(r_home, 'bin')
    r_bin64 = os.path.join(r_home, 'bin', 'x64')

    os.environ['R_HOME'] = r_home
    # Prepend R bin dirs to PATH so rpy2 finds Rscript etc.
    existing_path = os.environ.get('PATH', '')
    os.environ['PATH'] = r_bin64 + ';' + r_bin + ';' + existing_path

    # Register DLL search directories so Windows can locate R.dll / Rblas.dll
    for d in [r_bin64, r_bin]:
        if os.path.isdir(d):
            os.add_dll_directory(d)

    print(f'R_HOME : {r_home}')
    print('Windows R DLL paths configured — rpy2 is ready to import.')
else:
    print(f'Platform: {sys.platform} — no Windows R path setup required.')


Platform: darwin — no Windows R path setup required.


In [ ]:
# import rpy2
from rpy2.robjects import r
# Call the package, and run the PRE-ATAP r-script

# The script, Articles dataset and Masterlist dataset should be placed in the proceeding folder
r.source("proceeding/Pre-ATAP.R")


In [9]:
# move file and rename columns
r.source("proceeding/move_and_rename_columns.R")

Reading file from proceeding folder...
Renaming columns...
Writing file to input folder...
Successfully moved and renamed columns!
File saved to: input/articles_with_mentions.csv 


value,[0]
visible,[10]


In [10]:
# ----------------------------------------------------------------
# STAGE 2: QUOTE EXTRACTION
# ----------------------------------------------------------------
#
# This stage identifies and extracts direct quotations from the
# pre-processed article text, producing an initial quote-level
# dataset.
#
# INPUT:
# - Cleaned article-level dataset from Stage 2
#
# PROCESS:
# - Apply an automated quotation detection procedure to the
#   article text (e.g., regex-based or custom quote extraction
#   routines)
# - Extract each direct quote along with relevant context
#   (e.g., surrounding sentences, paragraph, article ID)
# - Record the position of each quote within the article
#   (e.g., character offsets or sentence indices)
#
# OUTPUT:
# - Quote-level dataset where each row corresponds to a single
#   extracted quotation, linked to its source article.
# ----------------------------------------------------------------


In [11]:
# ----------------------------------------------------------------
# STAGE 2: QUOTE EXTRACTION
# ----------------------------------------------------------------
#
# Direct speech is identified using automated quotation
# detection
#

# QuotationTool
In this stage, you will use the *QuotationTool* to extract quotes from CSV files in batch. The tool processes all files in the `input` folder and generates individual CSV output files in the `output` folder for each input file. In addition to extracting the quotes, the tool also provides information about who the speakers are, the location of the quotes (and the speakers) within the text, the identified named entities, etc., which can be useful for your text analysis.  

**Note:** This code has been adapted (with permission) from the [GenderGapTracker GitHub page](https://github.com/sfu-discourse-lab/GenderGapTracker/tree/master/nlp/english) and modified to run on a Jupyter Notebook. The quotation tool's accuracy rate is evaluated in [this article](https://doi.org/10.1371/journal.pone.0245533).

<div class="alert alert-block alert-warning">
<b>User guide to using a Jupyter Notebook</b> 

If you are new to Jupyter Notebook, feel free to take a quick look at [this user guide](https://github.com/Australian-Text-Analytics-Platform/quotation-tool/blob/main/documents/jupyter-notebook-guide.pdf) for basic information on how to use a notebook.
</div>

### Quotation Tool User Guide

For instructions on how to use the Quotation Tool, please refer to the [Quotation Tool User Guide](documents/quotation_help_pages.pdf).

<div class="alert alert-block alert-warning">
<b>Installing Libraries</b> 

The requirements file <b>requirements.txt</b> is included with this notebook. Run Pip install -r requirements.txt under Python environment that running the Jupyter Notebook.

</div>

## 1. Setup
Before you begin, you need to import the QuotationTool and the necessary libraries and initiate them to run in this notebook.

In [ ]:
# import the QuotationTool

import warnings
from extract_display_quotes import QuotationTool

import nltk
nltk.download('punkt_tab')

# initialize the QuotationTool
qt = QuotationTool()

## 2. Prepare your input files
Your CSV files are already in the `input` folder after the stage 2. Each CSV file should contain a text column with the content you want to analyze for quotes.

<table style='margin-left: 10px'><tr>
<td> <img src='./img/csv_icon.png' style='width: 45px'/> </td>
</tr></table>


<div class="alert alert-block alert-warning">
<b>Processing large files</b> 
    
Processing large CSV files or many files may take some time. Be patient and monitor the progress in the output messages. As a guideline, for a corpus with a file size of 54.13 MB (~26,000 newspaper articles in plain text format), it can take ca 45 minutes to extract quotes.
</div>

## 3. Process files and extract quotes

Run the cell below to process all CSV files in the `input` folder. The tool will:
1. Read each CSV file from the `input` folder
2. Extract quotes from the text content
3. Identify speakers and named entities
4. Generate a separate CSV output file for each input file in the `output` folder

<div class="alert alert-block alert-info">
<b>Tools:</b>    
    
- nltk: for sentence tokenization
- spaCy: for text cleaning, normalisation, and named entity recognition
- quote_extractor: for extracting quotes and speakers
- pandas: for reading and writing CSV files
</div>

<div class="alert alert-block alert-warning">
<b>Output Format</b> 
    
Each output CSV file will contain the extracted quotes with columns for quote content, speaker, verb, indices, entities, and other metadata. The output files will be saved in the `output` folder with names corresponding to the input files.
</div>

In [ ]:
# Process all CSV files in the input folder and generate output CSV files
qt.process_files()

## 4. Understanding the output

Once processing is complete, you'll find CSV files in the `output` folder. Each output file corresponds to an input file and contains the extracted quotes with detailed information.

<div class="alert alert-block alert-info">
<b>What's included in the output:</b>    

The output CSV files contain the following information:
- **Quote extraction**: Quotes identified using syntactic and heuristic rules
- **Speaker identification**: Who said each quote
- **Named entities**: Entities identified within quotes and speakers using spaCy
- **Metadata**: Quote locations, types, and other useful information
    
<b>Note:</b> This tool uses spaCy to tokenize the text, which initially splits the text into tokens based on whitespace characters, and then applies language specific rules to further refine the outcome. For example, the word "don't" does not contain whitespace, but would be split into two tokens: "do" and "n't", whereas "U.K." would remain as one token. For more information about spaCy tokenizer, please visit [this page](https://spacy.io/usage/linguistic-features#tokenization).
</div>

<div class="alert alert-block alert-danger">
<b>Memory limitation in Binder</b> 
    
The free Binder deployment is only guaranteed a maximum of 2GB memory. Processing very large text files may cause the session (kernel) to re-start due to insufficient memory. Check [the user guide](https://github.com/Sydney-Informatics-Hub/HASS-29_Quotation_Tool/blob/main/documents/jupyter-notebook-guide.pdf) for more info. 
</div>

<div class="alert alert-block alert-warning">
<b>What information is included in the output CSV files?</b> 

In general, the quotes are extracted either based on syntactic or heuristic rules. Some quotes can be stand-alone in a sentence, or followed by another quote (floating quote) from the same speaker. Please refer to [this document](https://doi.org/10.1371/journal.pone.0245533.s001) for further information about the quote extraction process.  

**Columns in the output CSV:**
    
**text_id:** the unique ID of the text.
    
**text_name:** the name of the text from the input file.
    
**quote_id/speaker_id:** the unique ID of the extracted quote/speaker.
    
**quote/speaker:** the content of the extracted quote and the speaker.
    
**verb:** the verb used to determine the extracted quote.
    
**quote_index/speaker_index/verb_index:** the location of the first and the last characters of the extracted quote/speaker/verb in the text.
    
**quote_entities/speaker_entities:** the entity name and type of the entities identified in the extracted quote/speaker.
    
**quote_token_count:** the length of the extracted quote (in characters).
    
**quote_type:** the type of quote based on how it is extracted.
    
**floating_quote:** whether the extracted quote is a floating quote, i.e., a follow up quote from the same speaker (The value TRUE here means that the quote is a floating quote, while FALSE means that the quote is not a floating quote).

**Quotation symbols:** Q (Quotation mark), S (Speaker), V (Verb), C (Content).  

**Named Entities:**  PERSON (People, including fictional), NORP (Nationalities or religious or political groups), FAC (Buildings, airports, highways, etc.), ORG (Companies, agencies, institutions, etc.), GPE (Countries, cities, states), LOC (Non-GPE locations, mountain ranges, bodies of water).
</div>

## 5. Access your results

Your extracted quotes are now available as CSV files in the `output` folder. Each input file has a corresponding output file containing all the extracted quotes and their metadata.

<div class="alert alert-block alert-success">
<b>Output Location:</b> 
    
All output CSV files are saved in the `output` folder in your workspace directory. You can access them directly from there.
</div>

<div class="alert alert-block alert-info">
<b>What's in each output file:</b>

Each output CSV file contains the full results with the following information:
+ **Quote and speaker information**: The extracted quotes, their speakers, and the verbs used
+ **Entity information**: Named entities identified in quotes and speakers (ORG, PERSON, GPE, NORP, FAC, LOC)
+ **Metadata**: Quote indices, quote types, token counts, and whether quotes are floating quotes
+ **Quote types**: Different quotation types (e.g., SVC, Heuristic) as identified by the GenderGapTracker quote extractor

For detailed information about each column, refer to section 4 above.
</div>

<div class="alert alert-block alert-warning">
<b>Note about the data:</b>

- Words are case sensitive in the output (e.g., "said", "Said", and "SAID" are treated as different)
- Each row represents one extracted quote with all associated information
- Entity names and types are listed when identified by spaCy
- You can perform your own frequency calculations and analyses using the CSV files with tools like Excel, Python pandas, or R
</div>

In [14]:
# ----------------------------------------------------------------
# STAGES 3–5: POST-ATAP PROCESSING
# ----------------------------------------------------------------
# At this point in the workflow, the quotation tool has been
# run. The resulting quote-level extraction files are
# expected to be present in the designated output directory.
#
# These files contain the raw quotations identified in the
# article texts, together with article identifiers and limited
# attribution information. The following post-ATAP stages take
# these raw outputs and transform them into a validated,
# statement-level dataset.
#
# 3. Speaker Identification and Coreference Resolution
#    Extracted quotes are linked to speakers using named-entity
#    recognition and rule-based coreference resolution. This step
#    assigns each statement to a unique interest group where
#    possible (e.g., via matching to a masterlist using a stable
#    identifier such as `uniqid`).
#
# 4. Statement Validation and Filtering
#    Automated speaker assignments are filtered and validated to
#    remove misattributions, ambiguous speakers, and non-substantive
#    quotations. The output of this step is a high-confidence
#    statement dataset suitable for replication and analysis.
#
# 5. Enrichment with Surrounding Text
#    Each retained quote is stored together with surrounding textual
#    context (e.g., sentence window, paragraph, or other local
#    context from the source article) to preserve interpretive
#    meaning and support downstream coding and robustness checks.
#
# OUTPUT (written to /output by the post-ATAP pipeline):
# - Final statement-level dataset (one row per validated statement)
#
# ----------------------------------------------------------------


In [15]:
# Call the package, and run the PRE-ATAP r-script

# The script, Articles dataset and Masterlist dataset should be placed in the proceeding folder
r.source("proceeding/Post-ATAP.R")

=== POST-ATAP: BUILDING STATEMENTS FROM QUOTES ===

Loading ATAP quote extractions from /output...
  Reading CSV: articles_with_mentions_quotes.csv 
Loaded 80 quotes from 1 ATAP file(s)
Post-ATAP summary:
  n_quotes         = 80 
  n_articles_subset = 12 

Processing quote entities...
Processed 30 entity matches
Identifying relevant quotes...
Found 39 relevant quotes
Creating statements dataset...
Created 6 final statements

=== POST-ATAP PIPELINE COMPLETE ===
Final output: 6 statements created
Statements saved to /output:
 - output/final_statements.csv 


value,[0]
visible,[10]
